# PneumoniaMNIST 上的 SSME-Lite

七个已发布的 PneumoniaMNIST 分类器先在官方 test split 上推理，再用 20 个可见标签和 400 个无标签样本估计 AUC。

交互式报告在 [pneumoniamnist_report.html](pneumoniamnist_report.html)。10 个 seed 的对比在 `experiments/pneumoniamnist/results/nl20_nu400_m7/`。


## 这次估计的 AUC 排名

下面读的是已经保存的 `experiments/pneumoniamnist/results/demo_report/metrics.csv`，对应 `random_state=0` 的那一次划分。区间是潜在标签不确定性，不是跨 seed 的标准误。


In [1]:
from pathlib import Path
import pandas as pd

metrics = pd.read_csv(Path('experiments/pneumoniamnist/results/demo_report/metrics.csv'))
auc = metrics.loc[metrics.metric == 'auc', ['model', 'estimate', 'interval_low', 'interval_high']]
auc = auc.sort_values('estimate', ascending=False).reset_index(drop=True)
auc.index = auc.index + 1
print(auc.to_string(float_format=lambda x: f'{x:.4f}'))


             model  estimate  interval_low  interval_high
1  automl_vision_3    0.9853        0.9749         0.9942
2  automl_vision_1    0.9848        0.9753         0.9945
3  automl_vision_2    0.9841        0.9729         0.9933
4      autokeras_2    0.9826        0.9758         0.9904
5      autokeras_3    0.9740        0.9638         0.9836
6      autokeras_1    0.9735        0.9642         0.9823
7    resnet18_28_1    0.9733        0.9627         0.9825


## 从推理到报告

重新跑需要能下载 MedMNIST 权重，并且安装 `ai-edge-litert` 来读三个 AutoML Vision 模型。脚本和下面这段是同一条路径。

```bash
python experiments/pneumoniamnist/pneumonia_ssme.py
```


In [ ]:
from pathlib import Path
import sys
from ssme_lite import PredictionMatrixGenerator, SSMEEstimator, SemiSupervisedSplit

ROOT = Path('experiments/pneumoniamnist')
sys.path.insert(0, str(ROOT))
from pneumonia_models import (
    MODELS, build_models, check_against_official, download_weights, load_test_images, save_scores,
)

images, y = load_test_images()
models = build_models(download_weights())
generator = PredictionMatrixGenerator({name: models[name] for name in MODELS})
scores = generator.fit_transform(images)
check_against_official(models, y)
save_scores(scores, y, generator.model_names_)

splitter = SemiSupervisedSplit(20, 400, random_state=0)
parts = splitter.indices(len(y))
estimator = SSMEEstimator(max_iter=100, early_stopping=False, random_state=0)
estimator.fit(
    scores[parts['estimation']],
    splitter.partial_labels(y, parts),
    model_names=generator.model_names_,
)
report = estimator.report(
    n_draws=100,
    title='PneumoniaMNIST',
    primary_metric='auc',
    sample_ids=parts['estimation'],
)
report.save(ROOT / 'results' / 'demo_report')
print(report.ranking('auc').to_string(index=False))


七个模型是 `resnet18_28_1`、三份 AutoKeras 和三份 AutoML Vision。划分由 `SemiSupervisedSplit` 固定：估计池里 20 个标签、400 个无标签，其余 test 样本留作 held-out。迁移学习是另一条链，见 `experiments/transfer/transfer_ssme.py`。
